CALCULAR 
- Población en edad de trabajar
- población que no está en edad de trabajar
- Población económicamente activa
- Población económicamente inactiva
- Tasa de ocupación
- Tasa de desempleo

In [0]:
#Leyendo la tabla de caracteristicas generales
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
ft = spark.read.format("delta").table("workspace.electiva.ft")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")

# Convertir el DataFrame de Spark a Pandas
cgse = cgse.toPandas()
desempleados=desempleados.toPandas()
ft=ft.toPandas()
ocupados=ocupados.toPandas()

In [0]:
#POBLACION EN EDAD DE TRABAJAR 
PET = ft[ft['PET']==1]['FEX_C18'].sum()
PET

In [0]:
#POBLACION QUE NO ESTA EN EDAD DE TRABAJAR
NO_PET = cgse[(cgse['P6040'] < 15)]['FEX_C18'].sum()
NO_PET

In [0]:
#POBLACION ECONOMICAMENTE ACTIVA 
PEA = ft[ft['FT']==1]['FEX_C18'].sum()
PEA

In [0]:
#POBLACION ECONOMICAMENTE INACTIVA 
PEI = ft[ft['PET']==1 & (ft['P6240'].isin([3, 4, 5, 6]))]['FEX_C18'].sum()
PEI

In [0]:
#TASA DE DESOCUPACION
D = desempleados[desempleados['DSI'] == 1]['FEX_C18'].sum()
TD = (D / PEA)*100
print(f'Tasa de desocupacion:  {TD:.2f} %')

In [0]:
#TASA DE OCUPACION
O = PEA - D
TO = (O/PET)*100
print(f'Tasa de ocupacion:  {TO:.2f} %')



TASA DE DESEMPLEO PARA HOMBRES Y MUJERES EN TODOS LOS DEPARTAMENTOS

In [0]:
import pandas as pd
#Primero hay que anexar P3271 a la tabla de desempleados
desempleados = pd.merge(desempleados, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

In [0]:
import pandas as pd

# Nombres de los departamentos
departamento = {
    5: 'Antioquia',
    8: 'Atlántico',
    11: 'Bogotá D.C.',
    13: 'Bolívar',
    15: 'Boyacá',
    17: 'Caldas',
    18: 'Caquetá',
    19: 'Cauca',
    20: 'Cesar',
    23: 'Córdoba',
    25: 'Cundinamarca',
    27: 'Chocó',
    41: 'Huila',
    44: 'La Guajira',
    47: 'Magdalena',
    50: 'Meta',
    52: 'Nariño',
    54: 'Norte de Santander',
    63: 'Quindío',
    66: 'Risaralda',
    68: 'Santander',
    70: 'Sucre',
    73: 'Tolima',
    76: 'Valle del Cauca',
    81: 'Arauca',
    85: 'Casanare',
    86: 'Putumayo',
    88: 'San Andrés y Providencia',
    91: 'Amazonas',
    94: 'Guainía',
    95: 'Guaviare',
    97: 'Vaupés',
    99: 'Vichada'
}

# Filtrar personas desempleadas
desocupados = desempleados[desempleados['DSI'] == 1]

# Calcular mujeres desempleadas por departamento
MD = desocupados[desocupados['P3271'] == 2].groupby(
    'DPTO'
)['FEX_C18'].sum()

# Calcular hombres desempleados por departamento
HD = desocupados[desocupados['P3271'] == 1].groupby(
    'DPTO'
)['FEX_C18'].sum()

# Crear tabla de resultados
cantidad_desempleados = pd.DataFrame({
    'Mujeres desempleadas': MD,
    'Hombres desempleados': HD
}).fillna(0)

# Agregar nombres de los departamentos
cantidad_desempleados = cantidad_desempleados.reset_index()
cantidad_desempleados['Departamento'] = (
    cantidad_desempleados['DPTO'].map(departamento)
)

# Organizar la tabla
cantidad_desempleados = cantidad_desempleados[
    ['Departamento', 'Mujeres desempleadas', 'Hombres desempleados']
]

display(cantidad_desempleados.round(0))

In [0]:
import pandas as pd
## Primero hay que añadir la P3271 al modulo de fuerza de trabajo 
ft = pd.merge(ft, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],
                       on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

In [0]:
# Calcular la población por sexo en cada departamento

# 1. Mujeres
PEAM = ft[
    (ft['FT'] == 1) & (ft['P3271'] == 2)
].groupby('DPTO')['FEX_C18'].sum()

# 2. Hombres
PEAH = ft[
    (ft['FT'] == 1) & (ft['P3271'] == 1)
].groupby('DPTO')['FEX_C18'].sum()

# Nombres de los departamentos
departamento = {
    5: 'Antioquia',
    8: 'Atlántico',
    11: 'Bogotá D.C.',
    13: 'Bolívar',
    15: 'Boyacá',
    17: 'Caldas',
    18: 'Caquetá',
    19: 'Cauca',
    20: 'Cesar',
    23: 'Córdoba',
    25: 'Cundinamarca',
    27: 'Chocó',
    41: 'Huila',
    44: 'La Guajira',
    47: 'Magdalena',
    50: 'Meta',
    52: 'Nariño',
    54: 'Norte de Santander',
    63: 'Quindío',
    66: 'Risaralda',
    68: 'Santander',
    70: 'Sucre',
    73: 'Tolima',
    76: 'Valle del Cauca',
    81: 'Arauca',
    85: 'Casanare',
    86: 'Putumayo',
    88: 'San Andrés y Providencia',
    91: 'Amazonas',
    94: 'Guainía',
    95: 'Guaviare',
    97: 'Vaupés',
    99: 'Vichada'
}

# Crear tabla con los resultados
pea_departamentos = pd.DataFrame({
    'Mujeres': PEAM,
    'Hombres': PEAH
}).fillna(0)

pea_departamentos.index.name = 'DPTO'
pea_departamentos = pea_departamentos.reset_index()

# Reemplazar códigos por nombres
pea_departamentos['Departamento'] = pea_departamentos['DPTO'].map(departamento)

# Mostrar resultados
pea_departamentos = pea_departamentos[
    ['Departamento', 'Mujeres', 'Hombres']
]

display(pea_departamentos.round(0))

In [0]:
# 1. Calcular hombres desempleados por departamento
HD = desempleados[
    (desempleados['P3271'] == 1) &
    (desempleados['DSI'] == 1)
].groupby('DPTO')['FEX_C18'].sum()

# 2. Calcular mujeres desempleadas por departamento
MD = desempleados[
    (desempleados['P3271'] == 2) &
    (desempleados['DSI'] == 1)
].groupby('DPTO')['FEX_C18'].sum()

# 3. Calcular la PEA por sexo y departamento
PEAH = ft[
    (ft['FT'] == 1) & (ft['P3271'] == 1)
].groupby('DPTO')['FEX_C18'].sum()

PEAM = ft[
    (ft['FT'] == 1) & (ft['P3271'] == 2)
].groupby('DPTO')['FEX_C18'].sum()

# 4. Calcular las tasas de desempleo
TDH = (HD / PEAH) * 100
TDM = (MD / PEAM) * 100

# 5. Crear tabla de resultados
tasas = pd.DataFrame({
    'Desempleados hombres': HD,
    'PEA hombres': PEAH,
    'Tasa desempleo hombres (%)': TDH,
    'Desempleadas mujeres': MD,
    'PEA mujeres': PEAM,
    'Tasa desempleo mujeres (%)': TDM
}).reset_index()

# 6. Agregar nombres de departamentos
tasas['Departamento'] = tasas['DPTO'].map(departamento)

# 7. Mostrar tabla final
tasas = tasas[[
    'Departamento',
    'Desempleados hombres',
    'PEA hombres',
    'Tasa desempleo hombres (%)',
    'Desempleadas mujeres',
    'PEA mujeres',
    'Tasa desempleo mujeres (%)'
]]

display(tasas.round(2))